In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool

In [32]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()


In [4]:
len(docs)

9

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)


In [6]:
len(splitted_docs)

26

In [22]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(splitted_docs, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3422.02it/s]


In [34]:
@tool
def retriever_tool(query: str) :
    """This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports."""

    docs = vector_store.similarity_search(query)
    context = ""
    for doc in docs:
        context += doc.page_content + "\n"
    return context

In [24]:
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

In [35]:
System_Prompt = """You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge. """

In [36]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt)


In [37]:
query= " what is patient name and doctor name in the report?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})
result = response["messages"][-1].content

In [38]:
print(result)

**Patient name:** Ms. Nikita Chudhary  
**Doctor name:** Dr. Nitin Nahar  

These details are listed in the report header:

- “Ms. Nikita Chudhary” appears under the **Name** field.  
- “DR NITIN NAHAR” appears next to the report date, indicating the attending physician.
